# Scale-invariant Barcelona model — training (distances + centrality only)

The ablation study showed Barcelona's noise signal lives almost entirely in the 10
**distance + centrality** features — but those are exactly the features that go out-of-distribution
when the model is transferred to another city (their raw values depend on network size / urban fabric:
closeness shrinks as a network grows, `dist_to_trunk` explodes in big cities). This notebook builds a
**scale-invariant** version of those 10 features and trains Barcelona classification + regression models
on them, so the signal-rich features can also transfer.

## How the scale-invariant features are built

For each feature `f` in the 10, **within the city**, replace the raw value by its percentile rank:

```
df[f + '_pct'] = df[f].rank(pct=True)     # empirical CDF: fraction of the city's streets with value <= this one
```

The percentile is **dimensionless and city-relative**: "this street is in the 90th percentile of closeness
*for its city*" means the same thing in Barcelona, Berlin or Zaragoza regardless of absolute scale or
segment count. A model that learns "high closeness-percentile ⇒ louder" then applies correctly everywhere.
The 6 distances keep their meaning too (a street on a trunk ⇒ `dist_to_trunk` = 0 ⇒ low percentile in every
city). Because the rank transform is **monotonic**, it does not change anything *within* Barcelona — the
held-out scores here should match the ablation's raw `only-distances+centrality` model (RF ≈ 0.751 acc /
0.696 R²). The payoff is in transfer, tested in `02_test_all_cities_scale_invariant.ipynb`.

## Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)
import xgboost as xgb

## The 10 features and the percentile transform

In [2]:
FEATURES_10 = [
    'dist_to_trunk', 'dist_to_primary', 'dist_to_secondary',
    'dist_to_tertiary', 'dist_to_residential', 'dist_to_living_street',
    'betweenness', 'closeness_global', 'closeness_400', 'straightness',
]
PCT_COLS = [f + '_pct' for f in FEATURES_10]

def add_pct(df):
    """Add per-city percentile-rank versions of the 10 features (scale-invariant)."""
    out = df.copy()
    for f in FEATURES_10:
        out[f + '_pct'] = out[f].rank(pct=True)
    return out

print('scale-invariant feature columns:')
PCT_COLS

scale-invariant feature columns:


['dist_to_trunk_pct',
 'dist_to_primary_pct',
 'dist_to_secondary_pct',
 'dist_to_tertiary_pct',
 'dist_to_residential_pct',
 'dist_to_living_street_pct',
 'betweenness_pct',
 'closeness_global_pct',
 'closeness_400_pct',
 'straightness_pct']

## Load Barcelona and build the scale-invariant features

In [3]:
cls = add_pct(pd.read_csv("../../notebooks/_elena/data/bcn_noise_class_ml_dataset.csv").dropna())
reg = add_pct(pd.read_csv("../../notebooks/_elena/data/bcn_noise_regre_ml_dataset.csv").dropna())
print('class:', cls.shape, '| regre:', reg.shape)

# the percentile features are identical between the two datasets (same segments/features) -> one scaler
scaler_si = StandardScaler().fit(cls[PCT_COLS])
print('percentile feature summary (should be ~uniform on (0,1]):')
cls[PCT_COLS].describe().loc[["min", "mean", "max"]].round(3)

class: (12854, 32) | regre: (12854, 32)
percentile feature summary (should be ~uniform on (0,1]):


,dist_to_trunk_pct,dist_to_primary_pct,dist_to_secondary_pct,dist_to_tertiary_pct,dist_to_residential_pct,dist_to_living_street_pct,betweenness_pct,closeness_global_pct,closeness_400_pct,straightness_pct
min,0.002,0.035,0.098,0.166,0.408,0.148,0.004,0.0,0.0,0.0
mean,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.5,0.5,0.5
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.0,1.0,1.0


## Save the scaler and feature list

In [4]:
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

with open(f'{MODELS_DIR}/Sscaler_si.pkl', 'wb') as f:
    pickle.dump(scaler_si, f)
with open(f'{MODELS_DIR}/feature_columns_si.pkl', 'wb') as f:
    pickle.dump(PCT_COLS, f)
print('saved scaler + feature list')

saved scaler + feature list


## Train classification models (`noise_day` class)

In [5]:
Xc = scaler_si.transform(cls[PCT_COLS])
yc = cls['noise_day'].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42)

class_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'XGBoost': xgb.XGBClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
}
class_files = {'Logistic Regression': 'logreg_class_si.pkl',
               'XGBoost': 'xgb_class_si.pkl', 'Random Forest': 'rf_class_si.pkl'}

rows = []
for name, model in class_models.items():
    model.fit(Xc_tr, yc_tr)
    yp = model.predict(Xc_te)
    rows.append({'model': name,
                 'accuracy': accuracy_score(yc_te, yp),
                 'macro_f1': f1_score(yc_te, yp, average='macro'),
                 'within_1': np.mean(np.abs(yp - yc_te) <= 1)})
    with open(f'{MODELS_DIR}/{class_files[name]}', 'wb') as f:
        pickle.dump(model, f)

pd.DataFrame(rows).set_index('model').round(4)

,accuracy,macro_f1,within_1
model,,,
Logistic Regression,0.5994,0.3758,0.9825
XGBoost,0.7417,0.7405,0.9961
Random Forest,0.7480,0.7305,0.9949


## Train regression models (`noise_day` dB)

In [6]:
Xr = scaler_si.transform(reg[PCT_COLS])
yr = reg['noise_day'].to_numpy(dtype=float)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=42)

regre_models = {
    'Linear Regression': LinearRegression(),
    'XGBoost': xgb.XGBRegressor(),
    'Random Forest': RandomForestRegressor(random_state=42),
}
regre_files = {'Linear Regression': 'linreg_regre_si.pkl',
               'XGBoost': 'xgb_regre_si.pkl', 'Random Forest': 'rf_regre_si.pkl'}

rows = []
for name, model in regre_models.items():
    model.fit(Xr_tr, yr_tr)
    yp = model.predict(Xr_te)
    rows.append({'model': name,
                 'r2': r2_score(yr_te, yp),
                 'mae': mean_absolute_error(yr_te, yp),
                 'rmse': np.sqrt(mean_squared_error(yr_te, yp))})
    with open(f'{MODELS_DIR}/{regre_files[name]}', 'wb') as f:
        pickle.dump(model, f)

pd.DataFrame(rows).set_index('model').round(4)

,r2,mae,rmse
model,,,
Linear Regression,0.4159,4.2682,5.4030
XGBoost,0.6787,3.0928,4.0075
Random Forest,0.6943,2.9322,3.9092


## Sanity check — scale-invariance must not change Barcelona

Percentile rank is monotonic, so within Barcelona the scale-invariant model should score the same as the
raw `only-distances+centrality` model from the ablation study (**RF 0.751 accuracy / 0.696 R²**,
XGBoost 0.742 / 0.679). If these match, the transform preserved all the Barcelona signal and any change
seen on the other cities is pure transfer improvement.

In [7]:
print("Barcelona held-out — scale-invariant 10-feature model")
print("  expect ~ raw only-distances+centrality: RF 0.751 acc / 0.696 R2, XGB 0.742 / 0.679")
print("  (above tables)")

Barcelona held-out — scale-invariant 10-feature model
  expect ~ raw only-distances+centrality: RF 0.751 acc / 0.696 R2, XGB 0.742 / 0.679
  (above tables)
